##### 랭크래프의 메모리 기능 활용하기
- 12_1에서 만든 챗봇은 이전 대화 내용을 기억하게 하려면 매번 수동으로 업데이트해야 했다. 
- 랭그래프에서 제공하는 메모리 기능을 활용하면 대화 내용을 간편하게 저장할 수 있다. 

In [3]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')


from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-5.6-luna")

from typing import Annotated # annotated는 타입 힌트를 사용할 때 사용하는 함수
from typing_extensions import TypedDict # TypedDict는 딕셔너리 타입을 정의할 때 사용하는 함수

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class State(TypedDict):
    """
    State 클래스는 TypedDict를 상속받습니다.

    속성:
        messages (Annotated[list[str], add_messages]): 메시지들은 "list" 타입을 가집니다.
       'add_messages' 함수는 이 상태 키가 어떻게 업데이트되어야 하는지를 정의합니다.
        (이 경우, 메시지를 덮어쓰는 대신 리스트에 추가합니다)
    """
    messages: Annotated[list[str], add_messages]

# StateGraph 클래스를 사용하여 State 타입의 그래프를 생성합니다.
graph_builder = StateGraph(State)


def generate(state: State):
    """
    주어진 상태를 기반으로 챗봇의 응답 메시지를 생성합니다.

    매개변수:
    state (State): 현재 대화 상태를 나타내는 객체로, 이전 메시지들이 포함되어 있습니다.
		
    반환값:
    dict: 모델이 생성한 응답 메시지를 포함하는 딕셔너리. 
          형식은 {"messages": [응답 메시지]}입니다.
    """ 
    return {"messages": [model.invoke(state["messages"])]}

graph_builder.add_node("generate", generate)

graph_builder.add_edge(START, "generate")
graph_builder.add_edge("generate", END)    

from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()

config = {"configurable": {"thread_id": "abcd"}} # 대화방 ID같은 것. thread_id가 유지된다면 기존 대화 내용을 계속 보존한 상태로 진행가능.

graph = graph_builder.compile(checkpointer=memory)

#------------ 여기서부터 달라진 코드가 있습니다.   
#from langchain.schema import HumanMessage
from langchain_core.messages import HumanMessage

while True:
    user_input = input("You\t:")
    
    if user_input in ["exit", "quit", "q"]:
        break
	#②
    for event in graph.stream({
        "messages": [HumanMessage(content=user_input)]}, 
        config,
        stream_mode="values"
    ):
        event["messages"][-1].pretty_print()

    print(f'\n현재 메시지 갯수: {len(event["messages"])}\n-------------------\n')

================================ Human Message =================================

안녕 난 김솔이야!
================================== Ai Message ==================================

안녕, 김솔! 만나서 반가워 😊 무엇을 도와줄까?

현재 메시지 갯수: 2
-------------------

================================ Human Message =================================

내 이름이 뭐라고했지?
================================== Ai Message ==================================

김솔이라고 했어! 😊

현재 메시지 갯수: 4
-------------------

